In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("insurance.csv")

print(df.head())
print("\nRandom sample:")
print(df.sample(5, random_state=42))

In [ ]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nUnique values:")
print(df.nunique())

In [ ]:
print("Mean charges:", df["charges"].mean())
print("Median charges:", df["charges"].median())
print("Minimum charges:", df["charges"].min())
print("Maximum charges:", df["charges"].max())

In [ ]:
print(df["smoker"].value_counts())

print("\nPercentages:")
print(df["smoker"].value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(data=df, x="charges", bins=30, kde=True)

plt.title("Distribution of Medical Insurance Charges")
plt.xlabel("Insurance Charges")
plt.ylabel("Number of Individuals")

plt.show()

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(data=df, x="smoker")

plt.title("Smoker vs. Non-Smoker")
plt.xlabel("Smoking Status")
plt.ylabel("Number of Individuals")

plt.show()

In [ ]:
X_reg = df.drop(columns=["charges"])
y_reg = df["charges"]

print("Features:", X_reg.shape)
print("Target:", y_reg.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

print("Training set:", X_train_reg.shape)
print("Testing set:", X_test_reg.shape)

In [ ]:
numerical_features_reg = X_reg.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_reg = X_reg.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", numerical_features_reg)
print("Categorical features:", categorical_features_reg)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_reg = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features_reg),
    ("categorical", categorical_pipeline, categorical_features_reg)
])

In [ ]:
X_train_reg_processed = preprocessor_reg.fit_transform(X_train_reg)
X_test_reg_processed = preprocessor_reg.transform(X_test_reg)

from sklearn.linear_model import LinearRegression

regression_model = LinearRegression()

regression_model.fit(X_train_reg_processed, y_train_reg)

y_pred_reg = regression_model.predict(X_test_reg_processed)

In [ ]:
regression_results = pd.DataFrame({
    "Actual": y_test_reg,
    "Predicted": y_pred_reg
})

print(regression_results.head(10))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test_reg, y_pred_reg, alpha=0.6)

min_value = min(y_test_reg.min(), y_pred_reg.min())
max_value = max(y_test_reg.max(), y_pred_reg.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    "r--"
)

plt.xlabel("Actual Insurance Charges")
plt.ylabel("Predicted Insurance Charges")
plt.title("Actual vs. Predicted Insurance Charges")

plt.show()

In [ ]:
X_cls = df[["age", "sex", "bmi", "children", "region"]]

y_cls = df["smoker"].map({
    "no": 0,
    "yes": 1
})

print(y_cls.value_counts())

In [ ]:
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

print("Training set:", X_train_cls.shape)
print("Testing set:", X_test_cls.shape)

print("\nTraining class proportions:")
print(y_train_cls.value_counts(normalize=True))

print("\nTesting class proportions:")
print(y_test_cls.value_counts(normalize=True))

In [ ]:
numerical_features_cls = X_cls.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_cls = X_cls.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", numerical_features_cls)
print("Categorical features:", categorical_features_cls)

In [ ]:
numerical_pipeline_cls = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline_cls = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_cls = ColumnTransformer([
    ("numerical", numerical_pipeline_cls, numerical_features_cls),
    ("categorical", categorical_pipeline_cls, categorical_features_cls)
])

In [ ]:
X_train_cls_processed = preprocessor_cls.fit_transform(X_train_cls)
X_test_cls_processed = preprocessor_cls.transform(X_test_cls)

from sklearn.linear_model import LogisticRegression

classification_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

classification_model.fit(
    X_train_cls_processed,
    y_train_cls
)

In [ ]:
y_pred_cls = classification_model.predict(X_test_cls_processed)

y_prob_cls = classification_model.predict_proba(
    X_test_cls_processed
)[:, 1]

classification_results = pd.DataFrame({
    "Actual": y_test_cls,
    "Predicted": y_pred_cls,
    "Probability_of_Smoking": y_prob_cls
})

print(classification_results.head(10))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

accuracy = accuracy_score(y_test_cls, y_pred_cls)

precision = precision_score(
    y_test_cls, y_pred_cls, zero_division=0
)

recall = recall_score(
    y_test_cls, y_pred_cls, zero_division=0
)

f1 = f1_score(
    y_test_cls, y_pred_cls, zero_division=0
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

print("\nClassification Report:")
print(
    classification_report(
        y_test_cls,
        y_pred_cls,
        labels=[0, 1],
        target_names=["Non-smoker", "Smoker"],
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(
    y_test_cls,
    y_pred_cls,
    labels=[0, 1]
)

cm_display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non-smoker", "Smoker"]
)

cm_display.plot(cmap="Blues")

plt.title("Confusion Matrix — Smoking Status")
plt.show()

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R²",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ],
    "Regression": [
        mae,
        rmse,
        r2,
        np.nan,
        np.nan,
        np.nan,
        np.nan
    ],
    "Classification": [
        np.nan,
        np.nan,
        np.nan,
        accuracy,
        precision,
        recall,
        f1
    ]
})

print(comparison.round(3).to_string(index=False))